# F1 Race Outcome Classification — SVM training + SHAP

All tunable parameters (paths, hyperparameter grid, train/validation
years, SHAP settings) live in `config.json`, next to this notebook —
edit that file, not the code below, to change any setting.

Same methodological decisions as the full model comparison: walk-forward
tuning inside the training window, single true holdout on the test year,
`class_weight='balanced'`, explicit cold-start handling
(`is_missing_<col>` flag + median imputation, fit only on the training
fold).

**Difference from the Random Forest SHAP notebook**: SVM is not a tree
model, so `shap.TreeExplainer` (exact, fast) does not apply. This uses
`shap.KernelExplainer` instead — model-agnostic, but approximate and much
slower. Two consequences, both handled explicitly below:
1. `SVC` needs `probability=True` to expose `predict_proba`
   (`KernelExplainer` needs a continuous output). This adds an internal
   5-fold calibration step and a small amount of extra randomness/cost
   on top of the SVM fit itself — declarable in the report, not a silent
   change of behavior.
2. The background dataset and the explained sample are both kept small
   (`shap_background_size`, `shap_n_explain` in the config) purely for
   runtime — directionally correct importance ranking is the goal here,
   not exact per-row SHAP values. Increase them if you have time/compute
   to spare.

## 0. Load configuration

Everything below reads from `config` — no hardcoded paths or
hyperparameters past this point.

In [1]:
import sys
print(sys.executable)

import numpy
print(numpy.__version__)
import json
import pandas as pd
import numpy as np

from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                              confusion_matrix)
import shap

CONFIG_PATH = "config.json"  # <-- expected to sit next to this notebook

with open(CONFIG_PATH) as f:
    config = json.load(f)

print("Loaded config:")
print(json.dumps(config, indent=2))

SENTINEL = -1

C:\Users\pietr\PycharmProjects\formula1_data_mining_project\.venv\Scripts\python.exe
2.4.6
Loaded config:
{
  "data_path": "C:/Users/pietr/PycharmProjects/formula1_data_mining_project/models_training/prediction.csv",
  "output_dir": "C:/Users/pietr/PycharmProjects/formula1_data_mining_project/models_training",
  "train_year_min": 2021,
  "train_year_max": 2024,
  "test_year": 2025,
  "walk_forward_validation_years": [
    2022,
    2023,
    2024
  ],
  "random_state": 42,
  "svm_param_grid": {
    "C": [
      0.0001,
      0.001,
      0.01,
      0.1,
      1,
      10,
      100
    ],
    "kernel": [
      "rbf",
      "linear"
    ],
    "gamma": [
      "scale",
      "auto"
    ]
  },
  "svm_max_iter": 10000,
  "shap_background_size": 30,
  "shap_n_explain": 60,
  "shap_nsamples": 100
}


## 1. Load data

In [2]:
df = pd.read_csv(config["data_path"])
print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Year range: {df.year.min()}-{df.year.max()}")

Loaded 2344 rows, 38 columns
Year range: 2021-2026


## 2. Cold-start sentinel handling

`COLD_START_COLS` is deliberately an explicit list in the code (not in
`config.json`, and not inferred by scanning for -1 values in the data) —
this list depends on how `prediction.csv` was built, not on
experiment-level settings, and scanning for -1 would incorrectly flag any
column where -1 happens to be a real, meaningful value (none here, but
being explicit avoids that class of bug on principle).

In [3]:
COLD_START_COLS = [
    "grid", "qualifying_position",
    "driver_std_points_prev", "driver_std_position_prev",
    "constructor_std_points_prev", "constructor_std_position_prev",
    "driver_avg_position_last3", "driver_avg_position_last5", "driver_avg_position_last10",
    "driver_podium_rate_last3", "driver_podium_rate_last5", "driver_podium_rate_last10",
    "driver_points_avg_last3", "driver_points_avg_last5", "driver_points_avg_last10",
    "constructor_avg_position_last3", "constructor_avg_position_last5",
    "driver_wins_at_circuit", "driver_avg_position_at_circuit",
    "teammate_h2h_avg_position_delta",
    "driver_dnf_rate_historical",
    "points_gap_to_leader",
    "constructor_change_flag",
    "days_since_last_race",
]

for col in COLD_START_COLS:
    flag_col = f"is_missing_{col}"
    df[flag_col] = (df[col] == SENTINEL).astype(int)
    df.loc[df[col] == SENTINEL, col] = np.nan

n_flagged = sum(df[f"is_missing_{c}"].sum() for c in COLD_START_COLS)
print(f"Total sentinel values converted to NaN + flagged: {n_flagged}")

Total sentinel values converted to NaN + flagged: 1271


## 3. Feature groups for the ColumnTransformer

In [4]:
categorical_cols = ["constructorId", "circuitId"]
binary_passthrough_cols = ["is_home_race", "is_home_constructor_race", "sprint_flag"] + \
                           [f"is_missing_{c}" for c in COLD_START_COLS]
exclude_cols = {"raceId", "driverId", "year", "round", "date", "target"}
numeric_cols = [c for c in df.columns
                if c not in exclude_cols and c not in categorical_cols
                and c not in binary_passthrough_cols]

print(f"Numeric (scaled+imputed): {len(numeric_cols)} cols")
print(f"Categorical (one-hot): {categorical_cols}")
print(f"Binary passthrough: {len(binary_passthrough_cols)} cols")

Numeric (scaled+imputed): 27 cols
Categorical (one-hot): ['constructorId', 'circuitId']
Binary passthrough: 27 cols


## 4. Train / test split

Years read from `config["train_year_min"]`, `config["train_year_max"]`,
`config["test_year"]` — change those in the JSON to shift the window,
nothing here needs editing.

In [5]:
train_mask = (df.year >= config["train_year_min"]) & (df.year <= config["train_year_max"])
test_mask = (df.year == config["test_year"])

X_train = df.loc[train_mask, numeric_cols + categorical_cols + binary_passthrough_cols]
y_train = df.loc[train_mask, "target"]
X_test = df.loc[test_mask, numeric_cols + categorical_cols + binary_passthrough_cols]
y_test = df.loc[test_mask, "target"]

test_year = config["test_year"]
print(f"Train: {len(X_train)} rows | Test ({test_year}): {len(X_test)} rows")
print("Train class distribution:\n", y_train.value_counts())
print("Test class distribution:\n", y_test.value_counts())

Train: 1799 rows | Test (2025): 479 rows
Train class distribution:
 target
no_points    899
points       630
podium       270
Name: count, dtype: int64
Test class distribution:
 target
no_points    239
points       168
podium        72
Name: count, dtype: int64


## 5. Walk-forward folds for hyperparameter tuning

Inside the training window only. Validation years come from
`config["walk_forward_validation_years"]`. Not random K-fold (breaks
temporal order), not nested CV (redundant — the true holdout year
already gives an unbiased final estimate).

In [6]:
train_years = df.loc[train_mask, "year"].reset_index(drop=True)
X_train_idx = X_train.reset_index(drop=True)
y_train_idx = y_train.reset_index(drop=True)

walk_forward_folds = []
for val_year in config["walk_forward_validation_years"]:
    tr_idx = np.where(train_years < val_year)[0]
    va_idx = np.where(train_years == val_year)[0]
    walk_forward_folds.append((tr_idx, va_idx))
    print(f"Fold (validate {val_year}): train={len(tr_idx)} rows, val={len(va_idx)} rows")

Fold (validate 2022): train=440 rows, val=440 rows
Fold (validate 2023): train=880 rows, val=440 rows
Fold (validate 2024): train=1320 rows, val=479 rows


## 6. Preprocessing pipeline

Imputer + scaler fit **only** on each training fold via `Pipeline` —
never on the full dataset.

In [7]:
preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ("bin", "passthrough", binary_passthrough_cols),
])

## 7. SVM — hyperparameter search

Grid comes from `config["svm_param_grid"]`; `max_iter` from
`config["svm_max_iter"]`. `probability=True` is added specifically to
support SHAP's `KernelExplainer` later (it needs `predict_proba`) — it
was not needed for the plain accuracy/macro-F1 evaluation on its own,
and adds a small amount of extra fitting cost (internal probability
calibration).

In [8]:
svm_estimator = SVC(class_weight="balanced", random_state=config["random_state"],
                     max_iter=config["svm_max_iter"], probability=True)
param_grid = {f"clf__{k}": v for k, v in config["svm_param_grid"].items()}

pipe = Pipeline([("prep", preprocessor), ("clf", svm_estimator)])
gs = GridSearchCV(pipe, param_grid, cv=walk_forward_folds, scoring="f1_macro", n_jobs=-1)
gs.fit(X_train_idx, y_train_idx)

print(f"Best params: {gs.best_params_}")
print(f"Best walk-forward macro-F1 (validation): {gs.best_score_:.3f}")

C:\Users\pietr\PycharmProjects\formula1_data_mining_project\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Best params: {'clf__C': 0.1, 'clf__gamma': 'scale', 'clf__kernel': 'linear'}
Best walk-forward macro-F1 (validation): 0.651


## Test performance (true holdout, evaluated once)

In [9]:
best_model = gs.best_estimator_
y_pred = best_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
f1m = f1_score(y_test, y_pred, average="macro")

print(f"Accuracy: {acc:.3f}")
print(f"Macro-F1: {f1m:.3f}")
print(classification_report(y_test, y_pred))
print("Confusion matrix (rows=true, cols=pred), classes:", sorted(y_test.unique()))
print(confusion_matrix(y_test, y_pred, labels=sorted(y_test.unique())))

Accuracy: 0.651
Macro-F1: 0.632
              precision    recall  f1-score   support

   no_points       0.74      0.76      0.75       239
      podium       0.55      0.90      0.68        72
      points       0.56      0.39      0.46       168

    accuracy                           0.65       479
   macro avg       0.62      0.68      0.63       479
weighted avg       0.65      0.65      0.64       479

Confusion matrix (rows=true, cols=pred), classes: ['no_points', 'podium', 'points']
[[181  12  46]
 [  2  65   5]
 [ 60  42  66]]


## 8. SHAP for SVM — KernelExplainer

Transform the data through the fitted preprocessor first, since
`KernelExplainer` needs the actual numeric feature matrix the classifier
sees (after scaling/one-hot/imputation), not the raw dataframe.

In [10]:
prep = best_model.named_steps["prep"]
clf = best_model.named_steps["clf"]

X_train_transformed = prep.transform(X_train_idx)
if hasattr(X_train_transformed, "toarray"):
    X_train_transformed = X_train_transformed.toarray()
X_test_transformed = prep.transform(X_test)
if hasattr(X_test_transformed, "toarray"):
    X_test_transformed = X_test_transformed.toarray()

feature_names = (numeric_cols +
                  list(prep.named_transformers_["cat"].get_feature_names_out(categorical_cols)) +
                  binary_passthrough_cols)

Background size, number of explained rows, and `nsamples` all come from
`config` (`shap_background_size`, `shap_n_explain`, `shap_nsamples`) —
purely to control runtime (`KernelExplainer` cost scales with
`background_size x n_explained x n_features x nsamples`). Increase them
in the JSON if you have time/compute to spare; the ranking should not
change much, only its precision.

In [11]:
background = shap.kmeans(X_train_transformed, config["shap_background_size"])
explain_sample = X_test_transformed[:config["shap_n_explain"]]

explainer = shap.KernelExplainer(clf.predict_proba, background)
shap_values = explainer.shap_values(explain_sample, nsamples=config["shap_nsamples"])

if isinstance(shap_values, list):
    mean_abs_shap = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
else:
    mean_abs_shap = np.abs(shap_values).mean(axis=(0, 2))

importance_df = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": mean_abs_shap
}).sort_values("mean_abs_shap", ascending=False)

print("TOP 20 FEATURES BY MEAN |SHAP VALUE|")
importance_df.head(20)

  0%|          | 0/60 [00:00<?, ?it/s]

TOP 20 FEATURES BY MEAN |SHAP VALUE|


,feature,mean_abs_shap
1,qualifying_position,0.087134
16,driver_points_avg_last10,0.029601
26,driver_seasons_in_f1,0.019943
18,constructor_avg_position_last5,0.018552
2,driver_age,0.014344
24,constructor_change_flag,0.013746
0,grid,0.013527
6,constructor_std_position_prev,0.010989
89,is_missing_teammate_h2h_avg_position_delta,0.010159
21,teammate_h2h_avg_position_delta,0.010009


## Save the full ranking

In [12]:
import os
out_path = os.path.join(config["output_dir"], "shap_svm_feature_importance.csv")
importance_df.to_csv(out_path, index=False)
print(f"Saved to {out_path}")

Saved to C:/Users/pietr/PycharmProjects/formula1_data_mining_project/models_training\shap_svm_feature_importance.csv
